# Notebook to extract 0.8s (TR) spaced image frames and audio clips from a movie stimulus for feature extraction

Use a Python environment with OpenCV (`cv2`). Optional downstream steps may use additional libraries.

## run video (images)

In [ ]:
# NATURALISTIC_PATH_SETUP
from pathlib import Path
import os
STIMULI = Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli'))
DATA = Path(os.environ.get('NATURALISTIC_ENCODING_DATA', '../data'))

import cv2
import os
import sys
import glob
import re


### despicable me

In [ ]:
stim='DM'
video_path=f'../data/{stim}.mp4'
save_dir=f'../data/{stim}_frames/'
TR=0.8 # how often to sample in seconds

video_capture = cv2.VideoCapture(video_path)

frame_rate = video_capture.get(cv2.CAP_PROP_FPS)
frames_to_skip = int(round(frame_rate*TR))

frame_count = 0
saved_frame_count = 0


while True:
    # Read the next frame from the video
    ret, frame = video_capture.read()

    if not ret:
        break  # Break the loop if we've reached the end of the video

    if frame_count % frames_to_skip == 0:
        output_frame_path = os.path.join(save_dir, f"frame_{saved_frame_count:04d}.jpg")
        cv2.imwrite(output_frame_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 100])  # Adjust the quality as needed
        saved_frame_count += 1

    frame_count += 1

# Release the video capture object
video_capture.release()
# logging.info(f"Saved {saved_frame_count} frames from {video_path} to {save_dir}")


#### get all frames too

In [ ]:
stim='DM'
video_path=f'../data/{stim}.mp4'
save_dir=f'../data/{stim}_frames_all/'

video_capture = cv2.VideoCapture(video_path)

frame_rate = video_capture.get(cv2.CAP_PROP_FPS)

frame_count = 0
saved_frame_count = 0


while True:
    # Read the next frame from the video
    ret, frame = video_capture.read()

    if not ret:
        break  # Break the loop if we've reached the end of the video

    output_frame_path = os.path.join(save_dir, f"frame_{frame_count:06d}.jpg")
    cv2.imwrite(output_frame_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 100])  # Adjust the quality as needed
    frame_count += 1

# Release the video capture object
video_capture.release()
# logging.info(f"Saved {saved_frame_count} frames from {video_path} to {save_dir}")


### The Present
#first convert TP.mp4 to a .mov since h264 codec not working with cv2
#ffmpeg -i TP.mp4 -vcodec libx264 TP.mov

In [ ]:
stim='TP'
video_path=f'../data/{stim}.mov'
save_dir=f'../data/{stim}_frames/'
TR=0.8 # how often to sample in seconds

video_capture = cv2.VideoCapture(video_path)

frame_rate = video_capture.get(cv2.CAP_PROP_FPS)
frames_to_skip = int(round(frame_rate*TR))

frame_count = 0
saved_frame_count = 0


while True:
    # Read the next frame from the video
    ret, frame = video_capture.read()

    if not ret:
        break  # Break the loop if we've reached the end of the video

    if frame_count % frames_to_skip == 0:
        output_frame_path = os.path.join(save_dir, f"frame_{saved_frame_count:04d}.jpg")
        cv2.imwrite(output_frame_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 100])  # Adjust the quality as needed
        saved_frame_count += 1

    frame_count += 1

# Release the video capture object
video_capture.release()
# logging.info(f"Saved {saved_frame_count} frames from {video_path} to {save_dir}")


### Friends (Cneuromod)

In [ ]:
from pathlib import Path
import os

friends_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends' / 's1')

for stim in ['friends_s01e01b','friends_s01e02a','friends_s01e02b']:
    video_path=f'{friends_path}/{stim}.mkv'
    save_dir=f'../data/{stim}_frames/'
    if not os.path.exists(save_dir):
        # Create the directory
        os.makedirs(save_dir)
        print(f"Directory {save_dir} created.")
    else:
        print(f"Directory {save_dir} already exists.")

    TR=1.49 # how often to sample in seconds

    video_capture = cv2.VideoCapture(video_path)

    frame_rate = video_capture.get(cv2.CAP_PROP_FPS)
    frames_to_skip = int(round(frame_rate*TR))

    frame_count = 0
    saved_frame_count = 0


    while True:
        # Read the next frame from the video
        ret, frame = video_capture.read()

        if not ret:
            break  # Break the loop if we've reached the end of the video

        if frame_count % frames_to_skip == 0:
            output_frame_path = os.path.join(save_dir, f"frame_{saved_frame_count:04d}.jpg")
            cv2.imwrite(output_frame_path, frame, [int(cv2.IMWRITE_JPEG_QUALITY), 100])  # Adjust the quality as needed
            saved_frame_count += 1

        frame_count += 1

    # Release the video capture object
    video_capture.release()
    # logging.info(f"Saved {saved_frame_count} frames from {video_path} to {save_dir}")


## split into 0.8 second video clips

In [ ]:
import os
import subprocess

def split_video(input_file, output_dir, segment_length=0.8):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Command to split the video using FFmpeg
    ffmpeg_command = [
        'ffmpeg',
        '-v', 'error',  # Verbose output for errors
        '-i', input_file,
        '-c:v', 'libx264',  # Use H.264 codec for video re-encoding
        '-c:a', 'aac',      # Use AAC codec for audio re-encoding
        '-b:v', '1000k',    # Set video bitrate
        '-b:a', '128k',     # Set audio bitrate
        '-force_key_frames', f'expr:gte(t,n_forced*{segment_length})',  # Force keyframes at segment_length intervals
        '-f', 'segment',
        '-segment_time', str(segment_length),
        '-reset_timestamps', '1',
        os.path.join(output_dir, 'output%03d.mp4')
    ]

    # Run the command and capture output
    result = subprocess.run(ffmpeg_command, capture_output=True, text=True)

    # Check for errors
    if result.returncode != 0:
        print("Error splitting video:")
        print(result.stderr)
        return

    print("Video split successfully.")

# Example usage
input_video = '../data/DM.mp4'
output_directory = '../data/DM_videos'
split_video(input_video, output_directory)

In [ ]:
#TP is a little messed up, do it a bit differently

import os
import subprocess

def split_video(input_file, output_dir, segment_length=0.8):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    # Command to split the video using FFmpeg
    ffmpeg_command = [
        'ffmpeg',
        '-v', 'error',  # Verbose output for errors
        '-i', input_file,
        '-c:v', 'libx264',  # Use H.264 codec for video re-encoding
        '-c:a', 'aac',      # Use AAC codec for audio re-encoding
        '-b:v', '500k',     # Lower video bitrate to reduce buffering
        '-b:a', '64k',      # Lower audio bitrate to reduce buffering
        '-force_key_frames', f'expr:gte(t,n_forced*{segment_length})',  # Force keyframes at segment_length intervals
        '-f', 'segment',
        '-segment_time', str(segment_length),
        '-reset_timestamps', '1',
        '-max_muxing_queue_size', '1024',  # Increase the muxing queue size
        os.path.join(output_dir, 'output%03d.mp4')
    ]

    # Run the command and capture output
    result = subprocess.run(ffmpeg_command, capture_output=True, text=True)

    # Check for errors
    if result.returncode != 0:
        print("Error splitting video:")
        print(result.stderr)
        return

    print("Video split successfully.")

# Example usage
# split_video(input_video, output_directory)


input_video = '../data/TP.mp4'
output_directory = '../data/TP_videos'
split_video(input_video, output_directory)

## run audio

In [ ]:
import librosa
import soundfile
import numpy as np


In [ ]:
#can use net2brain env
#first turn the mp4s into .wavs for convenience

In [ ]:
stim='TP'

audio_data, sr = librosa.load(f'../data/{stim}.wav', sr=20000)
clip_length = 2.0  # length of each clip in seconds
step_size = 0.8  # step size in seconds (the HBN fMRI TR)

# Calculate the number of samples for clip length and step size
clip_samples = int(clip_length * sr)
step_samples = int(step_size * sr)

# Number of clips
num_clips = int((len(audio_data) - clip_samples) / step_samples) + 1


for i in range(num_clips):
    start_sample = i * step_samples
    end_sample = start_sample + clip_samples

    # Extract clip
    clip = audio_data[start_sample:end_sample]

    # Save clip as .wav file
    #librosa.write_wav(f'../../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)
    soundfile.write(f'../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)

### friends
save files as .wavs first

In [ ]:
from pathlib import Path
import os
import subprocess

friends_path = str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends' / 's1')

stim='friends_s01e01a'
for stim in ['friends_s01e01b','friends_s01e02a','friends_s01e02b']:

    input_file=f'{friends_path}/{stim}.mkv'
    output_file=f'../data/{stim}.wav'

    command = [
        'ffmpeg',
        '-i', input_file,
        '-vn',
        '-acodec', 'pcm_s16le',
        '-ar', '44100',
        '-ac', '2',
        output_file
    ]

    try:
        subprocess.run(command, check=True)
        print(f"File converted and saved as {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e}")

In [ ]:
for stim in ['friends_s01e01a','friends_s01e01b','friends_s01e02a','friends_s01e02b']:
    
    if not os.path.exists(f'../data/{stim}_clips/'):
        os.makedirs(f'../data/{stim}_clips/')
    
    audio_data, sr = librosa.load(f'../data/{stim}.wav', sr=20000)
    clip_length = 2.0  # length of each clip in seconds
    step_size = 1.49  # step size in seconds (the HBN fMRI TR)

    # Calculate the number of samples for clip length and step size
    clip_samples = int(clip_length * sr)
    step_samples = int(step_size * sr)

    # Number of clips
    num_clips = int((len(audio_data) - clip_samples) / step_samples) + 1


    for i in range(num_clips):
        start_sample = i * step_samples
        end_sample = start_sample + clip_samples

        # Extract clip
        clip = audio_data[start_sample:end_sample]

        # Save clip as .wav file
        #librosa.write_wav(f'../../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)
        soundfile.write(f'../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)

#### process the next friends stim

In [ ]:
from pathlib import Path
import os
#for stim in ['friends_s01e01a','friends_s01e01b','friends_s01e02a','friends_s01e02b']:

for episode in range(3, 8):  # This will loop from 2 to 7 inclusive
    for variant in ['a', 'b']:
        stim = f"friends_s01e{episode:02d}{variant}"  # Zero-padded episode numbers

        if not os.path.exists(f'../data/{stim}_clips/'):
            os.makedirs(f'../data/{stim}_clips/')
        audio_data, sr = librosa.load(str(Path(os.environ.get('NATURALISTIC_ENCODING_STIMULI', '../stimuli')) / 'friends' / 's1' / f's1{stim}.wav'), sr=20000)
        clip_length = 2.0  # length of each clip in seconds
        step_size = 1.49  # step size in seconds (the HBN fMRI TR)
    
        # Calculate the number of samples for clip length and step size
        clip_samples = int(clip_length * sr)
        step_samples = int(step_size * sr)
    
        # Number of clips
        num_clips = int((len(audio_data) - clip_samples) / step_samples) + 1
    
    
        for i in range(num_clips):
            start_sample = i * step_samples
            end_sample = start_sample + clip_samples
    
            # Extract clip
            clip = audio_data[start_sample:end_sample]
    
            # Save clip as .wav file
            #librosa.write_wav(f'../../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)
            soundfile.write(f'../data/{stim}_clips/clip_{i:04d}.wav', clip, sr)